# Submission de los resultados
Este notebook está pensado para el final del reto. Se os compartirá un dataset para realizar predicciones con vuestro modelo.

Chequea que funciona el script con validation_dataset_sample.json antes de que te compartamos el dataset.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
import os
from datasets import Dataset
import pandas as pd
from dotenv import load_dotenv
from sklearn.metrics import classification_report

load_dotenv()

# Añadir src al path
sys.path.append(os.path.abspath(os.path.join('../src')))

from model_utils import *
from data_utils import *
from prompts import ABS_SYSTEM_PROMPT, ABSOLUTE_PROMPT
from submission import Submission

Cargamos el modelo finetuneado

In [ ]:
VALIDATION_INPUT_FILENAME = os.getenv("VALIDATION_INPUT_FILENAME", "../data/validation_dataset_sample.json")

MODEL_NAME = os.getenv("MODEL_NAME", "prometheus-eval/prometheus-7b-v2.0")
MODEL_FT_PATH = os.getenv("MODEL_FT_PATH", "../output/prometheus_finetuned")
PROMPT_COL = os.getenv("PROMPT_COL", "user_content")
OUTPUT_FILENAME = "../output/submission.json"

Los nombres de las variables del fichero de validación son los mismos que los del dataset_sample. Si has creado alguna función adicional que genere alguna variable derivada, ejecutala aquí para que genere esa variable

In [ ]:
# dataset
df = load_data(VALIDATION_INPUT_FILENAME)
dataset = Dataset.from_pandas(df) # si se usa datasets de HF


Genera el prompt que se pasará al modelo

Si tienes alguna especial, puedes añadirla aquí. P.e.
```
# puedes usar uno custom si lo prefieres 
def format_instruction_custom(sample,system_prompt,user_prompt, output_col = "user_content"):

    question = sample.get('question') or ''
    proposed_answer = sample.get('proposed_answer') or ''
    answer = sample.get('answer') or ''

    # Inyección en la plantilla de evaluación absoluta
    user_content = system_prompt + "\n\n" + absolute_prompt.format(
        question= context,
        answer=answer,
        proposed_answer= proposed_answer
    )
    
    return {output_col: user_content}
````



In [ ]:
# generamos los prompts
prompt_col="user_content"
dataset = dataset.map(format_instruction, fn_kwargs={"system_prompt": ABS_SYSTEM_PROMPT, "user_prompt":ABSOLUTE_PROMPT,"output_col":prompt_col})

Cargamos el modelo

In [ ]:
model, tokenizer = load_lora_model(MODEL_NAME, MODEL_FT_PATH)

In [ ]:
Prediciones

In [ ]:
#1. predicción para batches 

dataset = dataset.map(lambda batch: model_predict_batched(batch = batch, model = model, tokenizer = tokenizer,  input_col = prompt_col ), 
                      batched=True, batch_size=8)
dataset = dataset.map(split_model_reason_result)
dataset

In [ ]:
# OJO" Estas columnas deben de estar en el dataset final
output_cols = ["po_m_pred", "po_m_reason", "pt_m_pred", "pt_m_reason", "pg_m_pred", "pg_m_reason"]
        
if "id" in df_robustness_preds.column_names:
    output_cols.append("id")

Guardamos el fichero

In [ ]:
# se genera submission

save_data(dataset, OUTPUT_FILENAME)